# 04 - Synthetic Data Generation
**Fingo Income Predictor** | Tim CC26-PSU217

Input: `data/processed/survey_temporal_mapped.csv`  
Output:
- `data/synthetic/synthetic_52week_user_income.csv`
- `data/synthetic/synthetic_params.json`

Notebook ini membuat data sintetis 52 minggu untuk kebutuhan model Income Predictor. Data sintetis dibuat dari profil pendapatan survey yang sudah dibersihkan dan dipetakan secara temporal pada notebook sebelumnya.

In [1]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir("/content")

GITHUB_USERNAME = "ClarisyaA"
REPO_NAME       = "fingo-income-analysis"
BRANCH_NAME     = "feat/income-predictor-final"
LOCAL_DIR       = f"/content/{REPO_NAME}"
FRESH_CLONE     = False  # Set True hanya kalau mau clone ulang dari nol

def get_remote_url():
    try:
        token = userdata.get("GITHUB_TOKEN") if userdata else os.environ.get("GITHUB_TOKEN", "")
        if token:
            return f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", token
    except Exception:
        pass
    return f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, "***TOKEN***") if token else cmd

def run_cmd(cmd, check=True, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"$ {mask_cmd(cmd)}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {mask_cmd(cmd)}")
    return r

def remote_branch_exists():
    r = run_cmd(f"git ls-remote --heads {remote_url} {BRANCH_NAME}", check=False)
    return r.stdout.strip() != ""

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir("/content")
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f"git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}")
    else:
        run_cmd(f"git clone {remote_url} {LOCAL_DIR}")
        run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)
else:
    run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
    run_cmd("git fetch origin", cwd=LOCAL_DIR)
    if branch_exists:
        local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
        else:
            run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", cwd=LOCAL_DIR)
        run_cmd(f"git pull --rebase origin {BRANCH_NAME}", cwd=LOCAL_DIR)
    else:
        current_branch = run_cmd("git branch --show-current", check=False, cwd=LOCAL_DIR).stdout.strip()
        if current_branch != BRANCH_NAME:
            local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
            else:
                run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
print("\nRepo siap digunakan")
print(f"Working directory: {os.getcwd()}")
run_cmd("git branch --show-current", cwd=LOCAL_DIR)
run_cmd("git status --short", check=False, cwd=LOCAL_DIR)


$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
5457bb756c239058c1c01d7bd016accf8645e422	refs/heads/feat/income-predictor-final
$ git clone -b feat/income-predictor-final https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git /content/fingo-income-analysis
Cloning into '/content/fingo-income-analysis'...
Updating files:  68% (65/95)
Updating files:  69% (66/95)
Updating files:  70% (67/95)
Updating files:  71% (68/95)
Updating files:  72% (69/95)
Updating files:  73% (70/95)
Updating files:  74% (71/95)
Updating files:  75% (72/95)
Updating files:  76% (73/95)
Updating files:  77% (74/95)
Updating files:  78% (75/95)
Updating files:  80% (76/95)
Updating files:  81% (77/95)
Updating files:  82% (78/95)
Updating files:  83% (79/95)
Updating files:  84% (80/95)
Updating files:  85% (81/95)
Updating files:  86% (82/95)
Updating files:  87% (83/95)
Updating files:  88% (84/95)
Updating files:  89% (85/95)
Up

CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

In [2]:
# CELL 04.2 — Setup & import
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'scikit-learn', '--quiet'])

import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)
os.chdir("/content/fingo-income-analysis")

SIMULATION_YEAR   = 2026
N_SYNTHETIC_USERS = 3000
N_WEEKS           = 52

IDR_FMT = mticker.FuncFormatter(lambda x, _: f'Rp {x/1e6:.1f}jt' if x >= 1e6 else f'Rp {x/1e3:.0f}rb')

def fmt_idr(val):
    if pd.isna(val):
        return 'NaN'
    if val >= 1_000_000:
        return f'Rp {val/1_000_000:.2f}jt'
    return f'Rp {val/1_000:.0f}rb'

def ensure_dir(path):
    d = os.path.dirname(path) if '.' in os.path.basename(path) else path
    if d:
        os.makedirs(d, exist_ok=True)

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop('index', False), **kwargs)

def get_week_of_month(dt):
    return (dt.day - 1) // 7 + 1

ORDERED_GIG_TYPES = [
    'ojek_online', 'kurir', 'jualan_online', 'freelance_desain',
    'freelance_it', 'content_creator', 'tutor', 'pekerja_harian'
]

print('Setup selesai')


Setup selesai


In [3]:
# CELL 04.3 — Load survey data
df_feat = pd.read_csv('data/processed/survey_temporal_mapped.csv')
print(f'Dimuat: survey_temporal_mapped.csv ({df_feat.shape})')

GIG_MEDIAN_INCOME = {}
for gt in ORDERED_GIG_TYPES:
    subset = df_feat[df_feat['gig_type'] == gt]['avg_weekly_income']
    GIG_MEDIAN_INCOME[gt] = subset.median() if len(subset) > 0 else df_feat['avg_weekly_income'].median()

GIG_VOLATILITY = {
    'ojek_online': 0.35,
    'kurir': 0.30,
    'jualan_online': 0.45,
    'freelance_desain': 0.40,
    'freelance_it': 0.40,
    'content_creator': 0.50,
    'tutor': 0.20,
    'pekerja_harian': 0.35,
}

GIG_INCOME_CAP = {
    'ojek_online':      {'min_week': 50000, 'max_week': 600000},
    'kurir':            {'min_week': 50000, 'max_week': 550000},
    'jualan_online':    {'min_week': 0,     'max_week': 1500000},
    'freelance_desain': {'min_week': 0,     'max_week': 2000000},
    'freelance_it':     {'min_week': 0,     'max_week': 2500000},
    'content_creator':  {'min_week': 0,     'max_week': 2000000},
    'tutor':            {'min_week': 0,     'max_week': 900000},
    'pekerja_harian':   {'min_week': 50000, 'max_week': 700000},
}

income_cols_all = ['income_w1', 'income_w2', 'income_w3', 'income_w4']
GIG_CAP_P98 = {}

for gt in ORDERED_GIG_TYPES:
    subset = df_feat[df_feat['gig_type'] == gt]
    inc = subset[income_cols_all].values.flatten()
    inc = inc[(~np.isnan(inc)) & (inc > 0)]
    if len(inc) >= 10:
        p98_survey = np.percentile(inc, 98)
        form_max   = GIG_INCOME_CAP.get(gt, {}).get('max_week', 2000000)
        GIG_CAP_P98[gt] = max(p98_survey, form_max * 0.7)
    else:
        GIG_CAP_P98[gt] = GIG_INCOME_CAP.get(gt, {}).get('max_week', 1500000)

GLOBAL_CAP_P98 = np.nanpercentile(df_feat[income_cols_all].values.flatten(), 98)
fallback_bps = 500000 / 4

print('GIG_MEDIAN_INCOME dan GIG_CAP_P98 siap')
print('Median income per gig:')
for gt, val in GIG_MEDIAN_INCOME.items():
    print(f'  {gt}: {fmt_idr(val)}')


Dimuat: survey_temporal_mapped.csv ((384, 88))
GIG_MEDIAN_INCOME dan GIG_CAP_P98 siap
Median income per gig:
  ojek_online: Rp 366rb
  kurir: Rp 386rb
  jualan_online: Rp 315rb
  freelance_desain: Rp 532rb
  freelance_it: Rp 682rb
  content_creator: Rp 298rb
  tutor: Rp 382rb
  pekerja_harian: Rp 392rb


In [4]:
# CELL 04.4 — Seasonal functions
def get_synthetic_dates(year=SIMULATION_YEAR):
    start = pd.Timestamp(f'{year}-01-01')
    return [start + pd.to_timedelta(7*w, unit='D') for w in range(N_WEEKS)]

def is_ramadan_lebaran(dt):
    m, d = dt.month, dt.day
    return (m == 3 and d >= 1) or (m == 4 and d <= 15)

def is_harbolnas(dt):
    m, d = dt.month, dt.day
    return (m == 11 and d in [11]) or (m == 12 and d in [12]) or (m == 10 and d in [10])

def get_seasonal_event_type(dt, pref_ramadan, pref_harbolnas, pref_payday, pref_weekend, pref_natal=0):
    if is_ramadan_lebaran(dt) and pref_ramadan:
        return 'ramadan_lebaran'
    if is_harbolnas(dt) and pref_harbolnas:
        return 'harbolnas'
    if (dt.month == 12 and dt.day >= 20) and pref_natal:
        return 'christmas_year_end'
    if (dt.month == 1 and dt.day <= 7) and pref_natal:
        return 'new_year'
    if (dt.day <= 7 or dt.day >= 25) and pref_payday:
        return 'payday'
    if dt.dayofweek >= 5 and pref_weekend:
        return 'weekend_active'
    return 'normal'

GIG_SEASONAL_BOOST = {
    'jualan_online':    {'harbolnas': 0.30, 'payday': 0.20, 'promo_aplikasi': 0.15},
    'kurir':            {'harbolnas': 0.25, 'promo_aplikasi': 0.18, 'weekend_active': 0.12},
    'ojek_online':      {'weekend_active': 0.18, 'payday': 0.12, 'promo_aplikasi': 0.12},
    'freelance_it': {},
    'freelance_desain': {},
    'tutor': {},
    'content_creator':  {'harbolnas': 0.12, 'payday': 0.10},
    'pekerja_harian':   {'weekend_active': 0.15, 'harbolnas': 0.12},
}

def get_total_seasonal_multiplier(user_prefs, dt, gig_type):
    multiplier = 1.0

    pr  = user_prefs.get('pref_ramadan_lebaran', 0)
    ph  = user_prefs.get('pref_harbolnas', 0)
    pp  = user_prefs.get('pref_payday', 0)
    pw  = user_prefs.get('pref_weekend', 0)
    ppr = user_prefs.get('pref_promo_aplikasi', 0)
    pn  = user_prefs.get('pref_natal_tahun_baru', 0)

    if is_ramadan_lebaran(dt) and pr:
        multiplier += np.random.uniform(0.08, 0.28)
    if is_harbolnas(dt) and ph:
        multiplier += np.random.uniform(0.05, 0.22)
    if (dt.month == 12 and dt.day >= 20) and pn:
        multiplier += np.random.uniform(0.05, 0.15)
    if (dt.month == 1 and dt.day <= 7) and pn:
        multiplier += np.random.uniform(0.03, 0.10)
    if (dt.day <= 7 or dt.day >= 25) and pp:
        multiplier += np.random.uniform(0.05, 0.15)
    if dt.dayofweek >= 5 and pw:
        multiplier += np.random.uniform(0.03, 0.12)

    if ppr and dt.month in [3, 6, 9, 11, 12]:
        gb = GIG_SEASONAL_BOOST.get(gig_type, {}).get('promo_aplikasi', 0)
        if gb > 0:
            multiplier += np.random.uniform(0, gb)

    gig_boosts = GIG_SEASONAL_BOOST.get(gig_type, {})
    event = get_seasonal_event_type(dt, pr, ph, pp, pw, pn)
    if event in gig_boosts:
        multiplier += np.random.uniform(0, gig_boosts[event])

    if gig_type == 'tutor' and dt.month in [7, 12]:
        multiplier -= np.random.uniform(0.05, 0.18)

    return np.clip(multiplier, 0.40, 2.00)

print('Seasonal functions siap')


Seasonal functions siap


In [5]:
# CELL 04.5 — AR(1) income generation
def generate_user_52week_income(base_income, user_cv, gig_type, user_prefs, dates, cap_p98, seed=None):
    if seed is not None:
        np.random.seed(seed)

    gig_vol = GIG_VOLATILITY.get(gig_type, 0.30)
    effective_cv = np.clip(max(user_cv, gig_vol * 0.7), 0.15, 0.70)

    n = len(dates)
    incomes = np.zeros(n)

    trend_direction = np.random.choice([-1, 0, 0, 1], p=[0.15, 0.45, 0.25, 0.15])
    trend_magnitude = base_income * 0.0025 * trend_direction

    noise_init = np.random.normal(0, base_income * effective_cv * 0.25)
    incomes[0] = max(0, min(base_income + noise_init, cap_p98))

    for t in range(1, n):
        dt = dates[t]
        seasonal_mult = get_total_seasonal_multiplier(user_prefs, dt, gig_type)
        expected_t = base_income * seasonal_mult

        income_t = (
            0.65 * incomes[t-1] +
            0.30 * expected_t +
            0.05 * trend_magnitude * t
        )
        income_t += np.random.normal(0, base_income * effective_cv * 0.15)

        if np.random.rand() < 0.05:
            income_t *= np.random.uniform(0.2, 0.5)
        elif np.random.rand() < 0.04:
            income_t *= np.random.uniform(1.3, 1.8)
        elif gig_type in ['content_creator', 'freelance_it', 'freelance_desain'] and np.random.rand() < 0.06:
            income_t *= np.random.uniform(0.0, 0.2)

        incomes[t] = max(0, min(income_t, cap_p98))

    return incomes

print('generate_user_52week_income() siap')


generate_user_52week_income() siap


In [6]:
# CELL 04.6 — Generate 3.000 synthetic users
np.random.seed(42)

sampled_base_users = df_feat.sample(
    n=N_SYNTHETIC_USERS,
    replace=True,
    random_state=42
).reset_index(drop=True)

dates_2026 = get_synthetic_dates(SIMULATION_YEAR)

PREF_COLS_FOR_SYNTH = [
    'pref_ramadan_lebaran', 'pref_harbolnas', 'pref_payday',
    'pref_awal_bulan', 'pref_weekend', 'pref_promo_aplikasi',
    'pref_natal_tahun_baru'
]

synth_rows = []

for syn_idx, base_resp in sampled_base_users.iterrows():
    syn_uid = f'SYN_{syn_idx:06d}'
    gt  = base_resp.get('gig_type', 'pekerja_harian')
    dom = base_resp.get('domisili_code', 'jabodetabek')

    base_4w = [base_resp.get(f'income_w{w}', 0) or 0 for w in [1, 2, 3, 4]]
    base_4w_valid = [v for v in base_4w if v > 0]

    if base_4w_valid:
        mean_base = np.mean(base_4w_valid)
        cv_base = np.std(base_4w_valid) / mean_base if mean_base > 0 else 0.25
    else:
        mean_base = GIG_MEDIAN_INCOME.get(gt, 300000)
        cv_base = GIG_VOLATILITY.get(gt, 0.30)

    variation_factor = np.random.uniform(0.75, 1.35)
    base_income = max(mean_base * variation_factor, 50000)
    user_cv = np.clip(cv_base + np.random.normal(0, 0.05), 0.15, 0.70)

    hari_kerja = int(np.clip(base_resp.get('hari_kerja_per_minggu', 5) + np.random.randint(-1, 2), 1, 7))
    jam_kerja = int(np.clip(base_resp.get('jam_kerja_per_hari', 8) + np.random.randint(-1, 2), 1, 16))
    usia_syn = int(np.clip(base_resp.get('usia', 25) + np.random.randint(-3, 4), 17, 65))

    cap_p98 = GIG_CAP_P98.get(gt, GLOBAL_CAP_P98)
    user_prefs = {col: int(base_resp.get(col, 0) or 0) for col in PREF_COLS_FOR_SYNTH}

    synth_incomes = generate_user_52week_income(
        base_income, user_cv, gt, user_prefs, dates_2026, cap_p98, seed=42+syn_idx
    )

    pr  = user_prefs['pref_ramadan_lebaran']
    ph  = user_prefs['pref_harbolnas']
    pp  = user_prefs['pref_payday']
    pw  = user_prefs['pref_weekend']
    pn  = user_prefs['pref_natal_tahun_baru']
    ppr = user_prefs['pref_promo_aplikasi']
    pa  = user_prefs['pref_awal_bulan']

    for wi, dt in enumerate(dates_2026):
        event_type = get_seasonal_event_type(dt, pr, ph, pp, pw, pn)

        synth_rows.append({
            'synthetic_user_id': syn_uid,
            'source_respondent_id': f'R{int(base_resp.name):04d}',
            'dataset_type': 'synthetic_52w',
            'week_index': wi + 1,
            'week_start_date': dt.strftime('%Y-%m-%d'),
            'year': dt.year,
            'month': dt.month,
            'week_of_month': get_week_of_month(dt),
            'quarter': dt.quarter,
            'is_month_start': int(dt.day <= 7),
            'is_month_end': int(dt.day >= 24),
            'is_payday_period': int(dt.day <= 7 or dt.day >= 25),
            'is_weekend': int(dt.dayofweek >= 5),
            'is_ramadan_lebaran_period': int(is_ramadan_lebaran(dt)),
            'is_harbolnas_period': int(is_harbolnas(dt)),
            'is_christmas_year_end': int(dt.month == 12 and dt.day >= 20),
            'is_new_year': int(dt.month == 1 and dt.day <= 7),
            'seasonal_event_type': event_type,
            'gig_type': gt,
            'domisili_code': dom,
            'usia': usia_syn,
            'experience_months_log': float(base_resp.get('experience_months_log', 0) or 0),
            'hari_kerja_per_minggu': hari_kerja,
            'jam_kerja_per_hari': jam_kerja,
            'total_jam_seminggu': hari_kerja * jam_kerja,
            'bps_jasa_weekly': float(base_resp.get('bps_jasa_weekly', fallback_bps) or fallback_bps),
            'pref_awal_bulan': pa,
            'pref_payday': pp,
            'pref_weekend': pw,
            'pref_ramadan_lebaran': pr,
            'pref_natal_tahun_baru': pn,
            'pref_harbolnas': ph,
            'pref_promo_aplikasi': ppr,
            'synthetic_weekly_income': round(synth_incomes[wi], 0),
            'is_synthetic': 1,
            'synthetic_generation_note': f'AR1: base={base_income:.0f}, cv={user_cv:.3f}, gig={gt}',
        })

df_synth_raw = pd.DataFrame(synth_rows)

safe_to_csv(df_synth_raw, 'data/synthetic/synthetic_52week_user_income.csv')

print(f'Synthetic 52w: {df_synth_raw["synthetic_user_id"].nunique()} users x {N_WEEKS}w = {len(df_synth_raw):,} rows')
print(f'Income range: {df_synth_raw["synthetic_weekly_income"].min():,.0f} - {df_synth_raw["synthetic_weekly_income"].max():,.0f}')
print(df_synth_raw.head().to_string(index=False))


Synthetic 52w: 3000 users x 52w = 156,000 rows
Income range: 15 - 1,940,300
synthetic_user_id source_respondent_id  dataset_type  week_index week_start_date  year  month  week_of_month  quarter  is_month_start  is_month_end  is_payday_period  is_weekend  is_ramadan_lebaran_period  is_harbolnas_period  is_christmas_year_end  is_new_year seasonal_event_type         gig_type domisili_code  usia  experience_months_log  hari_kerja_per_minggu  jam_kerja_per_hari  total_jam_seminggu  bps_jasa_weekly  pref_awal_bulan  pref_payday  pref_weekend  pref_ramadan_lebaran  pref_natal_tahun_baru  pref_harbolnas  pref_promo_aplikasi  synthetic_weekly_income  is_synthetic                        synthetic_generation_note
       SYN_000000                R0000 synthetic_52w           1      2026-01-01  2026      1              1        1               1             0                 1           0                          0                    0                      0            1              payday freela

In [7]:
# CELL 04.7 — Simpan synthetic_params.json
SURVEY_INCOME_PROFILE = {}

for gt in ORDERED_GIG_TYPES:
    subset = df_feat[df_feat['gig_type'] == gt]
    inc = subset[['income_w1', 'income_w2', 'income_w3', 'income_w4']].values.flatten()
    inc = inc[inc > 0]

    SURVEY_INCOME_PROFILE[gt] = {
        'n': int(len(subset)),
        'mean': float(np.mean(inc)) if len(inc) > 0 else 300000,
        'median': float(np.median(inc)) if len(inc) > 0 else 300000,
        'std': float(np.std(inc)) if len(inc) > 0 else 100000,
        'cv': float(np.std(inc) / np.mean(inc)) if len(inc) > 0 and np.mean(inc) > 0 else 0.3,
        'p10': float(np.percentile(inc, 10)) if len(inc) > 0 else 150000,
        'p90': float(np.percentile(inc, 90)) if len(inc) > 0 else 600000,
        'cap_p98': float(GIG_CAP_P98.get(gt, 1500000)),
        'volatility': GIG_VOLATILITY.get(gt, 0.30),
    }

synthetic_params = {
    'version': 'v13',
    'n_synthetic_users': N_SYNTHETIC_USERS,
    'n_weeks': N_WEEKS,
    'simulation_year': SIMULATION_YEAR,
    'income_profile_per_gig': SURVEY_INCOME_PROFILE,
    'gig_volatility': GIG_VOLATILITY,
    'gig_cap_p98': {k: float(v) for k, v in GIG_CAP_P98.items()},
    'global_cap_p98': float(GLOBAL_CAP_P98),
    'ar1_config': {
        'ar_weight': 0.65,
        'expected_weight': 0.30,
        'trend_weight': 0.05,
        'shock_prob': 0.05,
        'high_demand_prob': 0.04,
        'low_income_freelance_prob': 0.06,
    },
}

ensure_dir('data/synthetic/')

with open('data/synthetic/synthetic_params.json', 'w', encoding='utf-8') as f:
    json.dump(synthetic_params, f, indent=2, ensure_ascii=False)

print('Disimpan: data/synthetic/synthetic_params.json')
print('Disimpan: data/synthetic/synthetic_52week_user_income.csv')


Disimpan: data/synthetic/synthetic_params.json
Disimpan: data/synthetic/synthetic_52week_user_income.csv


In [8]:
# CELL 04.8 — Quick validation output synthetic
print('=== Synthetic Output Validation ===')
print(f'Rows: {len(df_synth_raw):,}')
print(f'Users: {df_synth_raw["synthetic_user_id"].nunique():,}')
print(f'Weeks per user min/max: {df_synth_raw.groupby("synthetic_user_id")["week_index"].nunique().min()} / {df_synth_raw.groupby("synthetic_user_id")["week_index"].nunique().max()}')
print('\nIncome summary:')
print(df_synth_raw['synthetic_weekly_income'].describe().round(0).to_string())
print('\nGig distribution:')
print(df_synth_raw.groupby('gig_type')['synthetic_user_id'].nunique().sort_values(ascending=False).to_string())


=== Synthetic Output Validation ===
Rows: 156,000
Users: 3,000
Weeks per user min/max: 52 / 52

Income summary:
count     156000.0
mean      401178.0
std       305290.0
min           15.0
25%       183671.0
50%       322612.0
75%       522441.0
max      1940300.0

Gig distribution:
gig_type
pekerja_harian      498
jualan_online       496
freelance_desain    459
tutor               456
freelance_it        360
content_creator     277
ojek_online         229
kurir               225


In [9]:
# GIT PUSH — Rewrite output lama dengan output notebook terbaru lalu push ke GitHub
import os
import subprocess
from google.colab import userdata

LOCAL_DIR   = "/content/fingo-income-analysis"
BRANCH_NAME = "feat/income-predictor-final"
NOTEBOOK_NAME = "04_Synthetic_Data_Generation.ipynb"

# Folder/file yang mau di-rewrite di GitHub
REWRITE_PATHS = [
    "data",
    "outputs",
    "notebooks",
    NOTEBOOK_NAME,
]

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True, hide_token=True):
    display_cmd = cmd

    if hide_token:
        try:
            token = userdata.get("GITHUB_TOKEN")
            if token:
                display_cmd = display_cmd.replace(token, "***TOKEN***")
        except Exception:
            pass

    print(f"$ {display_cmd}")

    r = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )

    if r.stdout.strip():
        print(r.stdout.strip())

    if r.stderr.strip():
        err = r.stderr.strip()

        if hide_token:
            try:
                token = userdata.get("GITHUB_TOKEN")
                if token:
                    err = err.replace(token, "***TOKEN***")
            except Exception:
                pass

        print(err)

    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {display_cmd}")

    return r


# =========================
# 0. Setup Git Identity
# =========================
run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')


# =========================
# 1. Setup Remote dengan Token
# =========================
print("\n[1] Setup remote GitHub")

try:
    token = userdata.get("GITHUB_TOKEN")

    if not token:
        raise ValueError("GITHUB_TOKEN kosong / belum ada di Colab Secrets")

    remote_url = f"https://{token}@github.com/ClarisyaA/fingo-income-analysis.git"
    run_cmd(f'git remote set-url origin "{remote_url}"')

except Exception as e:
    print("[WARN] GITHUB_TOKEN tidak ditemukan atau bermasalah.")
    print("       Push bisa gagal kalau remote masih butuh autentikasi.")
    print(f"       Detail: {e}")


# =========================
# 2. Pastikan Branch Benar
# =========================
print("\n[2] Pastikan branch aktif benar")

current_branch = run_cmd("git branch --show-current", check=False).stdout.strip()

if current_branch != BRANCH_NAME:
    print(f"[INFO] Branch aktif sekarang: {current_branch}")
    print(f"[INFO] Pindah ke branch: {BRANCH_NAME}")

    run_cmd(f"git fetch origin {BRANCH_NAME}", check=False)

    checkout_result = run_cmd(f"git checkout {BRANCH_NAME}", check=False)

    if checkout_result.returncode != 0:
        run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", check=False)


# =========================
# 3. Cek Status Awal
# =========================
print("\n[3] Status awal")
run_cmd("git status --short", check=False)


# =========================
# 4. Hapus Versi Lama dari Git Index
# =========================
print("\n[4] Hapus versi lama dari Git index")

for path in REWRITE_PATHS:
    run_cmd(f'git rm -r --cached --ignore-unmatch "{path}"', check=False)


# =========================
# 5. Add Ulang Versi Terbaru
# =========================
print("\n[5] Add ulang versi terbaru dari Colab")

for path in REWRITE_PATHS:
    if os.path.exists(path):
        run_cmd(f'git add -A "{path}"', check=False)
    else:
        print(f"[SKIP] Path tidak ditemukan di local: {path}")


# =========================
# 6. Cek Status Setelah Rewrite
# =========================
print("\n[6] Status setelah rewrite")
run_cmd("git status --short", check=False)


# =========================
# 7. Commit
# =========================
print("\n[7] Commit perubahan")

commit_msg = f'feat(DS2): rewrite output terbaru dari {NOTEBOOK_NAME}'

commit_result = run_cmd(
    f'git commit -m "{commit_msg}"',
    check=False
)

if commit_result.returncode != 0:
    print("[INFO] Tidak ada perubahan baru untuk di-commit.")
else:
    print("[OK] Commit berhasil dibuat.")


# =========================
# 8. Fetch Remote Terbaru
# =========================
print("\n[8] Fetch remote terbaru")
run_cmd("git fetch origin", check=False)


# =========================
# 9. Push Rewrite ke GitHub
# =========================
print("\n[9] Push ke GitHub dengan rewrite aman")

run_cmd(
    f"git push --force-with-lease -u origin {BRANCH_NAME}",
    check=True
)

print("\n✓ Push rewrite berhasil!")
print(f"✓ Branch remote sekarang mengikuti output terbaru dari: {NOTEBOOK_NAME}")
print(f"✓ Branch: {BRANCH_NAME}")

$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Setup remote GitHub
$ git remote set-url origin "https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git"

[2] Pastikan branch aktif benar
$ git branch --show-current
feat/income-predictor-final

[3] Status awal
$ git status --short
M data/synthetic/synthetic_52week_user_income.csv

[4] Hapus versi lama dari Git index
$ git rm -r --cached --ignore-unmatch "data"
rm 'data/processed/income_features.csv'
rm 'data/processed/survey_clean.csv'
rm 'data/processed/survey_temporal_mapped.csv'
rm 'data/processed/survey_weekly_income_long.csv'
rm 'data/raw/.gitkeep'
rm 'data/raw/Rata-Rata Pendapatan Bersih Sebulan Pekerja Bebas Menurut Provinsi dan Lapangan Pekerjaan Utama, 2025.csv'
rm 'data/raw/Rata-Rata Pendapatan Bersih Sebulan Pekerja Informal Menurut Provinsi dan Lapangan Pekerjaan Utama (rupiah), 2025.csv'
rm 'data/raw/Rata-Rata_Pendapatan_Bersih_Sebulan_Pekerja_Bebas_Menurut_Provinsi_dan